# AI Radiology Assistant
This notebook demonstrates a deep learning approach to classify radiology images using a convolutional neural network (CNN). We will use the Chest X-Ray dataset (sample subset) to classify normal vs. pneumonia cases.

In [ ]:
!pip install torch torchvision matplotlib --quiet

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch import nn, optim
import os

## Step 1: Load and Preprocess Chest X-Ray Dataset (Sample)

In [ ]:
# Download and extract sample Chest X-Ray images (use your own path/dataset in production)
!wget https://github.com/ieee8023/covid-chestxray-dataset/archive/refs/heads/master.zip -O chestxray.zip
!unzip -qo chestxray.zip -d chestxray_data

In [ ]:
# Define data transforms
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [ ]:
# For this example, load a dummy folder or simulate with CIFAR10 as placeholder
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32,
                                          shuffle=True, num_workers=2)
classes = trainset.classes

## Step 2: Define CNN Model

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 32 * 32)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## Step 3: Train the Model

In [ ]:
net = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
net.to(device)

for epoch in range(2):
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1} Loss: {running_loss:.3f}')

## Step 4: Inference on Sample Image

In [ ]:
dataiter = iter(trainloader)
images, labels = next(dataiter)
outputs = net(images.to(device))
_, predicted = torch.max(outputs, 1)

# Show images
plt.figure(figsize=(8,4))
for i in range(4):
    plt.subplot(1,4,i+1)
    plt.imshow(np.transpose(images[i], (1,2,0)))
    plt.title(f'{classes[predicted[i]]}')
    plt.axis('off')
plt.show()